# Geometry and numerical sensitivity

MFDRO separates three scientific decisions: how frequency measures are scaled, how their center is constructed, and how dispersion around that center is weighted. This notebook varies those decisions on one fixed dataset and keeps random seeds common wherever comparisons require them.

The purpose is numerical understanding, not selection of a configuration from the largest or smallest signal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mfdro import FrequencySpec, MultiFrequencySignal, SignalConfig, build_frequency_measures

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#183b4e", "#008c82", "#d17a22", "#6f5b9a"]

## 1. Hold the empirical measures fixed

A compact four-asset panel keeps exact optimal transport inexpensive enough for a notebook while preserving a genuinely multivariate geometry.

In [ ]:
rng = np.random.default_rng(731)
dates = pd.bdate_range("2022-01-03", "2023-12-29")
market = rng.normal(0.00015, 0.008, size=(len(dates), 1))
sector = rng.normal(0.0, 0.004, size=(len(dates), 2))
returns = pd.DataFrame(
    np.column_stack(
        [
            market[:, 0] + sector[:, 0],
            0.8 * market[:, 0] + 0.7 * sector[:, 0],
            1.1 * market[:, 0] + sector[:, 1],
            0.6 * market[:, 0] - 0.5 * sector[:, 1],
        ]
    )
    + rng.normal(0.0, 0.003, size=(len(dates), 4)),
    index=dates,
    columns=["asset_A", "asset_B", "asset_C", "asset_D"],
)
FREQUENCIES = (
    FrequencySpec("daily", 1.0),
    FrequencySpec("weekly", 5.0, rule="W-FRI"),
    FrequencySpec("monthly", 21.0, rule="ME"),
)
measures = build_frequency_measures(returns, frequency_specs=FREQUENCIES)
pd.Series({name: len(frame) for name, frame in measures.items()}, name="observations")

## 2. Scaling and frequency weighting answer different questions

Power scaling expresses an effective horizon model. Realized-volatility scaling standardizes each asset inside each frequency. Frequency weights then determine how much each scaled empirical measure contributes to dispersion.

In [ ]:
base_config = SignalConfig.projected(
    frequency_specs=FREQUENCIES,
    n_projections=128,
    n_quantiles=128,
    random_state=20250301,
)
scalings = {
    "power h=0.5": {"scaling": "power", "scaling_exponent": 0.5},
    "power h=1.0": {"scaling": "power", "scaling_exponent": 1.0},
    "realized volatility": {"scaling": "realized_volatility"},
}
weightings = ["uniform", "sample_size", "log_sample_size"]
sensitivity = pd.DataFrame(index=scalings, columns=weightings, dtype=float)
for scaling_label, scaling_options in scalings.items():
    for weighting in weightings:
        config = base_config.with_updates(frequency_weighting=weighting, **scaling_options)
        sensitivity.loc[scaling_label, weighting] = (
            MultiFrequencySignal(config).estimate(measures, seed=271828).sqrt_rho
        )

sensitivity

In [ ]:
figure, axis = plt.subplots(figsize=(7.5, 3.6))
image = axis.imshow(sensitivity.to_numpy(), cmap="Blues", aspect="auto")
axis.set_xticks(range(len(weightings)), labels=weightings, rotation=20)
axis.set_yticks(range(len(scalings)), labels=list(scalings))
axis.set_title(r"Sensitivity of $\sqrt{\rho}$ to declared scientific choices")
for row in range(sensitivity.shape[0]):
    for column in range(sensitivity.shape[1]):
        axis.text(column, row, f"{sensitivity.iloc[row, column]:.4f}", ha="center", va="center")
figure.colorbar(image, ax=axis, label=r"$\sqrt{\rho}$")
figure.tight_layout()

## 3. Center weights are not dispersion weights

`barycenter_weights` determine the center, while `explicit_frequency_weights` determine the weighted dispersion around it. Keeping the vectors separate prevents a robustness weighting experiment from silently moving the center in the same way.

In [ ]:
weighted_config = base_config.with_updates(
    frequency_weighting="explicit",
    explicit_frequency_weights=(5.0, 2.0, 1.0),
    barycenter_weights=(1.0, 2.0, 5.0),
)
weighted_estimate = MultiFrequencySignal(weighted_config).estimate(measures, seed=271828)
weights = pd.DataFrame(
    {
        "dispersion weight": weighted_estimate.frequency_weights,
        "center weight": weighted_estimate.barycenter_weights,
    },
    index=weighted_estimate.frequencies,
)
weights

In [ ]:
axis = weights.plot.bar(figsize=(7, 3.4), color=COLORS[:2], rot=0)
axis.set(
    title="Two normalized weighting systems", xlabel="frequency", ylabel="weight", ylim=(0, 0.7)
)
plt.tight_layout()

## 4. Projection count controls Monte Carlo resolution

Projected calculations use random directions. The path below shows estimates across five fixed seeds as the number of directions increases. Convergence need not be monotone for an individual seed.

In [ ]:
projection_counts = [8, 16, 32, 64, 128, 256]
projection_seeds = [11, 22, 33, 44, 55]
convergence = pd.DataFrame(index=projection_counts, columns=projection_seeds, dtype=float)
for count in projection_counts:
    config = base_config.with_updates(n_projections=count)
    engine = MultiFrequencySignal(config)
    for seed in projection_seeds:
        convergence.loc[count, seed] = engine.estimate(measures, seed=seed).sqrt_rho

convergence.index.name = "n_projections"
convergence

In [ ]:
figure, axis = plt.subplots(figsize=(8, 3.7))
for seed in projection_seeds:
    axis.plot(convergence.index, convergence[seed], color="#8aa2ad", alpha=0.55, linewidth=1)
axis.plot(
    convergence.index,
    convergence.mean(axis=1),
    color=COLORS[0],
    marker="o",
    linewidth=2.2,
    label="mean across seeds",
)
axis.fill_between(
    convergence.index,
    convergence.mean(axis=1) - convergence.std(axis=1),
    convergence.mean(axis=1) + convergence.std(axis=1),
    color=COLORS[0],
    alpha=0.12,
    label="±1 standard deviation",
)
axis.set_xscale("log", base=2)
axis.set(title="Projected estimate stability", xlabel="random directions", ylabel=r"$\sqrt{\rho}$")
axis.legend(frameon=False)
figure.tight_layout()

## 5. Free-support center and distance evaluation

A free-support barycenter stores one multivariate center. Its support can be returned for inspection. Sliced distance projects discrepancies; exact distance solves discrete transport against that support and is usually more expensive.

In [ ]:
free_sliced_config = SignalConfig(
    frequency_specs=FREQUENCIES,
    barycenter="free_support",
    barycenter_size=12,
    barycenter_random_state=19,
    distance="sliced",
    n_projections=256,
    n_quantiles=128,
    barycenter_max_iter=15,
    random_state=20250301,
)
sliced_estimate = MultiFrequencySignal(free_sliced_config).estimate(
    measures, seed=271828, include_support=True
)
exact_config = free_sliced_config.with_updates(distance="exact")
exact_estimate = MultiFrequencySignal(exact_config).estimate(measures, include_support=True)

assert sliced_estimate.support is not None
assert sliced_estimate.support.shape == (12, returns.shape[1])
pd.DataFrame(
    [sliced_estimate.to_series(), exact_estimate.to_series()],
    index=["free support + sliced", "free support + exact"],
)[["rho", "sqrt_rho", "seed", "n_assets"]]

In [ ]:
scaled_measures = {spec.name: measures[spec.name] / np.sqrt(spec.horizon) for spec in FREQUENCIES}
figure, axis = plt.subplots(figsize=(7.2, 5.2))
for color, (name, frame) in zip(
    COLORS[: len(scaled_measures)], scaled_measures.items(), strict=True
):
    axis.scatter(
        frame["asset_A"],
        frame["asset_B"],
        s=13,
        alpha=0.28,
        color=color,
        label=name,
    )
axis.scatter(
    sliced_estimate.support[:, 0],
    sliced_estimate.support[:, 1],
    s=70,
    marker="X",
    color="#111111",
    label="barycenter support",
)
axis.set(
    title="First two coordinates of the scaled empirical geometry",
    xlabel="scaled return: asset_A",
    ylabel="scaled return: asset_B",
)
axis.legend(frameon=False)
figure.tight_layout()

## Interpretation

Differences across rows and columns above are consequences of explicit model choices, not numerical defects. A defensible study declares a primary configuration, checks projection stability and economically plausible alternatives, and retains common seeds when comparing projected variants. Exact transport is a validation mode for tractable problems; it is not automatically a superior estimator for every sample size.